# Official GIN+ on ZINC — standalone Colab runner

This notebook trains **GIN+** from Luo, Shi & Wu, *Can Classic GNNs Be Strong Baselines for Graph-level Tasks? Simple Architectures Meet Excellence* (ICML 2025; [arXiv:2502.09263](https://arxiv.org/abs/2502.09263)). It deliberately does **not** reimplement GIN+. It clones and pins the [official `LUOyk1999/GNNPlus` repository](https://github.com/LUOyk1999/GNNPlus), verifies the official ZINC configuration byte-for-byte, and launches the repository's own `main.py` with `configs/gine/zinc.yaml`.

Official/paper settings enforced here:

- GIN+ is the repository's `gine` implementation (GIN with edge features).
- ZINC 12k subset and its official 10k/1k/1k train/validation/test splits.
- 12 message-passing layers, hidden dimension 80, BN, residuals, FFN, RWSE-20, sum pooling.
- Batch size 32; AdamW; LR 0.001; weight decay 1e-5; cosine-with-warmup; 50 warmup epochs; 2,000 epochs; L1 loss.
- Expected trainable parameter count: **477,241** (paper Appendix Table 10).
- Best test MAE is selected at the best validation-MAE epoch, exactly as in the official trainer.

Reproducibility note: the paper reports mean ± s.d. over **five** independent runs, but the repository README's ZINC shell example asks `run.sh` for only two repeats. The repository's seed code starts at `seed 0` and increments by one. Therefore this notebook defaults to the paper-faithful, repository-defined schedule **0, 1, 2, 3, 4**. The paper does not separately print the five numeric seed IDs, so that schedule is the strongest reproducible interpretation available from the official artifacts. Change `N_RUNS` to `2` only if you specifically want to mirror the README example rather than the paper's five-run protocol.

Each seed is launched in a separate process with the untouched official config. This is equivalent to the official repeat loop for independent seeded runs, while retaining a distinct Drive-backed result directory and final official checkpoint for every seed. Completed seeds are skipped safely on rerun. An interrupted seed is restarted from epoch 0 because the official config has `train.auto_resume=False`; any partial output is archived rather than deleted.

In [ ]:
# Mount Drive and declare the complete experiment specification.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

OFFICIAL_REPO = "https://github.com/LUOyk1999/GNNPlus.git"
OFFICIAL_COMMIT = "0e02ad9acc2f1e54b5ad71c051bf5dfb1fcb4f28"
OFFICIAL_CONFIG_REL = Path("configs/gine/zinc.yaml")
OFFICIAL_CONFIG_SHA256 = "894816439156cf82b18212f52aa5947f62d73a18e18af9ba750d160e648d9320"
EXPECTED_PARAMS = 477_241

BASE_SEED = 0
N_RUNS = 5                 # Paper protocol: five independent runs.
SEEDS = tuple(range(BASE_SEED, BASE_SEED + N_RUNS))
CONSOLE_EPOCH_PERIOD = 10  # Full unfiltered logs are always saved to Drive.

DRIVE_ROOT = Path("/content/drive/MyDrive/ginplus_zinc_official")
RUNS_DIR = DRIVE_ROOT / "runs"
DATASET_DIR = DRIVE_ROOT / "datasets"
PROVENANCE_DIR = DRIVE_ROOT / "provenance"
REPO_DIR = Path("/content/GNNPlus_official")
VENV_DIR = Path("/content/gnnplus_py310")
PYTHON = VENV_DIR / "bin/python"

for path in (DRIVE_ROOT, RUNS_DIR, DATASET_DIR, PROVENANCE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Seeds:       ", SEEDS)
print("Drive root:  ", DRIVE_ROOT)
print("Dataset cache:", DATASET_DIR)
print("GPU requested: yes")

## Exact official software environment and source

The official README specifies Python 3.9/3.10, PyTorch 2.2.0, torchvision 0.17.0, torchaudio 2.2.0, PyTorch Geometric 2.3.1, and scikit-learn 1.4.0. Current Colab runtimes can use a newer system Python and PyTorch, so this cell creates an isolated managed Python 3.10 environment instead of replacing Colab's kernel packages.

The CUDA 11.8 wheels and the matching official PyG extension wheels are used exactly as documented by the repository. Auxiliary packages that the README names but does not version are compatibility-locked; the complete `pip freeze` is saved to Drive. The official repository is checked out detached at the pinned commit, and no source file is edited.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from textwrap import dedent


def run(cmd, *, cwd=None, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print("\n[cmd]", " ".join(cmd), flush=True)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=capture,
    )


# Install uv only as the Python-3.10 environment bootstrapper.
if shutil.which("uv") is None:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv")
assert UV, "uv installation did not expose an executable"

environment_spec = {
    "python": "3.10",
    "torch": "2.2.0+cu118",
    "torchvision": "0.17.0+cu118",
    "torchaudio": "2.2.0+cu118",
    "torch_geometric": "2.3.1",
    "scikit_learn": "1.4.0",
    "numpy": "1.26.4",
    "pyg_lib": "0.4.0+pt22cu118",
    "torch_scatter": "2.1.2+pt22cu118",
    "torch_sparse": "0.6.18+pt22cu118",
    "torch_cluster": "1.6.3+pt22cu118",
    "torch_spline_conv": "1.2.2+pt22cu118",
}
environment_key = hashlib.sha256(
    json.dumps(environment_spec, sort_keys=True).encode()
).hexdigest()
marker = VENV_DIR / ".ginplus_environment.json"
reuse_environment = False
if marker.exists() and PYTHON.exists():
    try:
        reuse_environment = json.loads(marker.read_text())["key"] == environment_key
    except Exception:
        reuse_environment = False

if not reuse_environment:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)  # Scoped to this notebook's /content venv.
    run([UV, "venv", "--python", "3.10", "--seed", VENV_DIR])
    pip = [PYTHON, "-m", "pip", "install", "--disable-pip-version-check"]

    run(pip + [
        "--index-url", "https://download.pytorch.org/whl/cu118",
        "--extra-index-url", "https://pypi.org/simple",
        "torch==2.2.0", "torchvision==0.17.0", "torchaudio==2.2.0",
    ])
    run(pip + [
        "numpy==1.26.4",
        "scipy==1.12.0",
        "scikit-learn==1.4.0",
        "torch_geometric==2.3.1",
        "fsspec==2024.2.0",
        "rdkit==2023.9.5",
        "pytorch-lightning==2.2.5",
        "torchmetrics==1.3.2",
        "yacs==0.1.8",
        "networkx==3.2.1",
        "tensorboardX==2.6.2.2",
        "ogb==1.3.6",
        "wandb==0.16.3",
        "setuptools<81",
        "protobuf<5",
    ])
    run(pip + [
        "--no-index",
        "--find-links", "https://data.pyg.org/whl/torch-2.2.0+cu118.html",
        "pyg_lib==0.4.0+pt22cu118",
        "torch_scatter==2.1.2+pt22cu118",
        "torch_sparse==0.6.18+pt22cu118",
        "torch_cluster==1.6.3+pt22cu118",
        "torch_spline_conv==1.2.2+pt22cu118",
    ])
    marker.write_text(json.dumps({"key": environment_key, "spec": environment_spec}, indent=2))
else:
    print(f"[env] Reusing verified environment at {VENV_DIR}")

# Fresh pinned checkout. Existing tracked changes cause a hard stop.
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout; remove or rename it.")
    run(["git", "clone", "--filter=blob:none", OFFICIAL_REPO, REPO_DIR])
else:
    origin = run(["git", "-C", REPO_DIR, "remote", "get-url", "origin"], capture=True).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != OFFICIAL_REPO.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Unexpected repository origin: {origin}")
    dirty = run(["git", "-C", REPO_DIR, "status", "--porcelain", "--untracked-files=no"], capture=True).stdout
    if dirty.strip():
        raise RuntimeError("Official checkout has tracked modifications; refusing to overwrite them.\n" + dirty)
    run(["git", "-C", REPO_DIR, "fetch", "origin", OFFICIAL_COMMIT])
run(["git", "-C", REPO_DIR, "checkout", "--detach", OFFICIAL_COMMIT])

actual_commit = run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture=True).stdout.strip()
if actual_commit != OFFICIAL_COMMIT:
    raise RuntimeError(f"Commit mismatch: {actual_commit} != {OFFICIAL_COMMIT}")

print("[source] Official checkout pinned and clean:", actual_commit)

## Reproducibility audit before training

This cell verifies the official YAML's SHA-256 digest and the critical scientific settings, checks the exact runtime versions and CUDA access, snapshots the pristine config, and records provenance on Drive. Training will also stop immediately if the official model reports a parameter count other than 477,241.

In [ ]:
import re

config_path = REPO_DIR / OFFICIAL_CONFIG_REL
config_bytes = config_path.read_bytes()
actual_config_sha = hashlib.sha256(config_bytes).hexdigest()
if actual_config_sha != OFFICIAL_CONFIG_SHA256:
    raise RuntimeError(
        "Official ZINC config digest changed: "
        f"{actual_config_sha} != {OFFICIAL_CONFIG_SHA256}"
    )

config_text = config_bytes.decode("utf-8")
required_config_fragments = (
    "format: PyG-ZINC",
    "name: subset",
    "node_encoder_name: TypeDictNode+RWSE",
    "times_func: range(1,21)",
    "dim_pe: 28",
    "batch_size: 32",
    "layer_type: gine",
    "layers_mp: 12",
    "dim_inner: 80",
    "ffn: True",
    "residual: True",
    "base_lr: 0.001",
    "max_epoch: 2000",
    "num_warmup_epochs: 50",
    "weight_decay: 1e-5",
)
missing = [item for item in required_config_fragments if item not in config_text]
if missing:
    raise RuntimeError(f"Official config semantic audit failed; missing: {missing}")

runtime_probe = run(
    [PYTHON, "-c", dedent('''
        import json, sys, torch, torch_geometric, sklearn, numpy
        info = {
            'python': sys.version.split()[0],
            'torch': torch.__version__,
            'torch_geometric': torch_geometric.__version__,
            'sklearn': sklearn.__version__,
            'numpy': numpy.__version__,
            'cuda_available': torch.cuda.is_available(),
            'cuda_runtime': torch.version.cuda,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        }
        print(json.dumps(info))
    ''')],
    capture=True,
)
runtime_info = json.loads(runtime_probe.stdout.strip().splitlines()[-1])
expected_versions = {
    "python": "3.10",
    "torch": "2.2.0+cu118",
    "torch_geometric": "2.3.1",
    "sklearn": "1.4.0",
    "numpy": "1.26.4",
}
for key, expected in expected_versions.items():
    actual = runtime_info[key]
    if key == "python":
        ok = actual.startswith(expected + ".")
    else:
        ok = actual == expected
    if not ok:
        raise RuntimeError(f"Runtime mismatch for {key}: {actual!r} != {expected!r}")
if not runtime_info["cuda_available"]:
    raise RuntimeError("CUDA is unavailable. In Colab select Runtime > Change runtime type > GPU.")

freeze = run([PYTHON, "-m", "pip", "freeze"], capture=True).stdout
freeze_path = PROVENANCE_DIR / "requirements_frozen.txt"
freeze_path.write_text(freeze)
pristine_config_path = PROVENANCE_DIR / "official_gine_zinc.yaml"
pristine_config_path.write_bytes(config_bytes)

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "paper": "arXiv:2502.09263 / ICML 2025",
    "official_repo": OFFICIAL_REPO,
    "official_commit": OFFICIAL_COMMIT,
    "official_config": str(OFFICIAL_CONFIG_REL),
    "official_config_sha256": actual_config_sha,
    "expected_parameters": EXPECTED_PARAMS,
    "seeds": list(SEEDS),
    "seed_basis": (
        "Paper reports 5 independent seeds; official main.py increments from base seed 0. "
        "Paper does not separately enumerate seed IDs; README ZINC example requests only 2 repeats."
    ),
    "runtime": runtime_info,
    "dataset_cache": str(DATASET_DIR),
    "runs_root": str(RUNS_DIR),
}
provenance_path = PROVENANCE_DIR / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2))

print(json.dumps(provenance, indent=2))
print("\n--- verified official configs/gine/zinc.yaml ---\n")
print(config_text)

## Train the five official seeded runs

The exact command shape for each independent run is:

```text
python -u /content/GNNPlus_official/main.py \
  --cfg /content/GNNPlus_official/configs/gine/zinc.yaml \
  --repeat 1 seed <SEED>
```

Only the current working directory differs between seeds. Local symlinks route the official relative `./datasets` and `./results` paths to Drive. The raw subprocess log, the official `results/logging.log`, per-split `stats.json`, `subset_result.txt`, and the final official checkpoint are retained for every seed.

The official config checkpoints every 100 epochs, keeps the last checkpoint, and does not auto-resume. If Colab disconnects, rerun the notebook: completed seeds are skipped; the partial seed is archived and restarted cleanly to preserve exact official training semantics.

In [ ]:
import math
import signal
import statistics
import time


def directory_has_content(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def parse_seed_result(result_file: Path, seed: int) -> float:
    if not result_file.exists():
        raise RuntimeError(f"Official result file is missing: {result_file}")
    pattern = re.compile(rf"\bseed_{seed}:.*?test_mae:\s*([0-9.eE+-]+)")
    matches = pattern.findall(result_file.read_text(errors="replace"))
    if not matches:
        raise RuntimeError(f"Could not parse seed {seed} test MAE from {result_file}")
    return float(matches[-1])


def stream_official_training(cmd, *, cwd: Path, raw_log: Path, expected_params: int):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["WANDB_MODE"] = "disabled"  # Config already has wandb.use=False.

    print("\n[official cmd]", " ".join(map(str, cmd)), flush=True)
    print("[official cwd]", cwd, flush=True)
    print("[raw log]", raw_log, flush=True)
    raw_log.parent.mkdir(parents=True, exist_ok=True)
    param_seen = None
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    with raw_log.open("a", buffering=1) as log_file:
        for line in proc.stdout:
            log_file.write(line)
            stripped = line.rstrip()

            parameter_match = re.search(r"Num parameters:\s*([0-9,]+)", stripped)
            if parameter_match:
                param_seen = int(parameter_match.group(1).replace(",", ""))
                print(stripped, flush=True)
                if param_seen != expected_params:
                    proc.terminate()
                    proc.wait(timeout=30)
                    raise RuntimeError(
                        f"Official model parameter mismatch: {param_seen} != {expected_params}"
                    )
                continue

            epoch_match = re.search(r"> Epoch\s+(\d+):", stripped)
            if epoch_match:
                epoch = int(epoch_match.group(1))
                if epoch < 3 or (epoch + 1) % CONSOLE_EPOCH_PERIOD == 0 or epoch == 1999:
                    print(stripped, flush=True)
                continue

            important = (
                "[*] Run ID",
                "Start from epoch",
                "Checkpoint found",
                "Avg time per epoch",
                "Total train loop time",
                "Task done",
                "[*] All done",
                "Downloading",
                "Processing",
            )
            if any(token in stripped for token in important):
                print(stripped, flush=True)

    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [str(x) for x in cmd])
    if param_seen is None:
        raise RuntimeError("Training ended without the official parameter-count log line.")
    return param_seen


completed = []
for seed in SEEDS:
    seed_root = RUNS_DIR / f"seed_{seed}"
    results_dir = seed_root / "results"
    completion_path = seed_root / "completed.json"
    seed_root.mkdir(parents=True, exist_ok=True)

    if completion_path.exists():
        prior = json.loads(completion_path.read_text())
        if (
            prior.get("official_commit") == OFFICIAL_COMMIT
            and prior.get("config_sha256") == OFFICIAL_CONFIG_SHA256
            and prior.get("seed") == seed
            and prior.get("parameter_count") == EXPECTED_PARAMS
        ):
            print(f"\n[skip] seed {seed} already completed: test MAE={prior['test_mae']}")
            completed.append(prior)
            continue
        raise RuntimeError(f"Completion manifest for seed {seed} belongs to another specification.")

    if directory_has_content(results_dir):
        archive_dir = seed_root / "incomplete_archives"
        archive_dir.mkdir(exist_ok=True)
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archived = archive_dir / f"results_{timestamp}"
        shutil.move(str(results_dir), str(archived))
        print(f"[recovery] Archived incomplete seed {seed} output to {archived}")
    results_dir.mkdir(parents=True, exist_ok=True)

    # Use a fast local cwd; symlink only the official relative data/output paths to Drive.
    work_dir = Path(f"/content/ginplus_zinc_seed_{seed}")
    if work_dir.exists():
        shutil.rmtree(work_dir)  # Scoped to this notebook's seed-specific /content directory.
    work_dir.mkdir(parents=True)
    os.symlink(DATASET_DIR, work_dir / "datasets", target_is_directory=True)
    os.symlink(results_dir, work_dir / "results", target_is_directory=True)

    raw_log = seed_root / f"official_stdout_seed{seed}.log"
    command = [
        PYTHON,
        "-u",
        REPO_DIR / "main.py",
        "--cfg",
        REPO_DIR / OFFICIAL_CONFIG_REL,
        "--repeat",
        "1",
        "seed",
        str(seed),
    ]
    started = datetime.now(timezone.utc)
    parameter_count = stream_official_training(
        command,
        cwd=work_dir,
        raw_log=raw_log,
        expected_params=EXPECTED_PARAMS,
    )
    finished = datetime.now(timezone.utc)

    test_mae = parse_seed_result(results_dir / "subset_result.txt", seed)
    checkpoints = sorted(results_dir.glob("ckpt/*.ckpt"))
    if not checkpoints:
        raise RuntimeError(f"No official checkpoint retained for completed seed {seed}")

    record = {
        "seed": seed,
        "test_mae": test_mae,
        "parameter_count": parameter_count,
        "official_commit": OFFICIAL_COMMIT,
        "config_sha256": OFFICIAL_CONFIG_SHA256,
        "started_utc": started.isoformat(),
        "finished_utc": finished.isoformat(),
        "elapsed_hours": (finished - started).total_seconds() / 3600.0,
        "results_dir": str(results_dir),
        "raw_log": str(raw_log),
        "checkpoints": [str(path) for path in checkpoints],
        "command": [str(x) for x in command],
    }
    completion_path.write_text(json.dumps(record, indent=2))
    completed.append(record)
    print(f"[complete] seed {seed}: test MAE={test_mae}; checkpoint={checkpoints[-1]}")

print(f"\nCompleted {len(completed)}/{len(SEEDS)} requested seeds.")

## Aggregate and save the official five-run result

The official aggregation helper uses NumPy's default population standard deviation (`ddof=0`), so this cell does the same. The paper's reference result for GIN+ on ZINC is **0.065 ± 0.004 MAE**. Exact equality is not asserted because GPU kernels and hardware can introduce small numerical variation.

In [ ]:
import csv

records = []
for seed in SEEDS:
    path = RUNS_DIR / f"seed_{seed}" / "completed.json"
    if not path.exists():
        raise RuntimeError(f"Seed {seed} has not completed: {path}")
    records.append(json.loads(path.read_text()))

maes = [float(record["test_mae"]) for record in records]
mean_mae = statistics.fmean(maes)
population_sd = math.sqrt(statistics.fmean([(x - mean_mae) ** 2 for x in maes]))

aggregate = {
    "model": "official GIN+ (repository config name: gine)",
    "dataset": "ZINC subset",
    "metric": "test MAE at best validation-MAE epoch",
    "seeds": list(SEEDS),
    "test_mae_by_seed": {str(r["seed"]): r["test_mae"] for r in records},
    "mean_test_mae": mean_mae,
    "population_sd_test_mae": population_sd,
    "paper_reference_mean": 0.065,
    "paper_reference_sd": 0.004,
    "official_commit": OFFICIAL_COMMIT,
    "config_sha256": OFFICIAL_CONFIG_SHA256,
    "parameter_count": EXPECTED_PARAMS,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
summary_json = DRIVE_ROOT / "aggregate_summary.json"
summary_json.write_text(json.dumps(aggregate, indent=2))

summary_csv = DRIVE_ROOT / "seed_results.csv"
with summary_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["seed", "test_mae", "elapsed_hours", "results_dir"])
    writer.writeheader()
    for record in records:
        writer.writerow({key: record[key] for key in writer.fieldnames})

print("\nOfficial GIN+ / ZINC result")
for record in records:
    print(f"  seed {record['seed']}: {record['test_mae']:.6f}")
print(f"  mean ± population s.d.: {mean_mae:.6f} ± {population_sd:.6f}")
print("  paper reference:         0.065000 ± 0.004000")
print("\nSaved:")
print(" ", summary_json)
print(" ", summary_csv)
print(" ", PROVENANCE_DIR / "provenance.json")
print(" ", PROVENANCE_DIR / "requirements_frozen.txt")